In [4]:
#!/usr/bin/env python3
"""
PILOT STEP 1: Sample exactly 20 prompts from the existing curated_prompts_600.parquet
Stratified sampling: 7 factual, 7 ambiguous, 6 adversarial.
Zero re-parsing. Zero empty-string risk.
"""

import pandas as pd
import numpy as np
import uuid
from pathlib import Path

np.random.seed(42)
DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
SOURCE_PATH = DATA_DIR / "data" / "prompts" / "curated_prompts_600.parquet"
OUTPUT_DIR = DATA_DIR / "data" / "pilot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 1. Load the validated 600-prompt dataset
# =============================================================================
print(f"📥 Loading: {SOURCE_PATH}")
assert SOURCE_PATH.exists(), f"❌ File not found: {SOURCE_PATH}"

full_df = pd.read_parquet(SOURCE_PATH)
print(f"✅ Loaded {len(full_df)} prompts")
print(f"📊 Distribution: {full_df['difficulty_type'].value_counts().to_dict()}")

# =============================================================================
# 2. Stratified sampling: 7 factual, 7 ambiguous, 6 adversarial
# =============================================================================
print("\n🎲 Stratified sampling 20 prompts...")

# Sample per type
factual_sample = full_df[full_df["difficulty_type"] == "factual"].sample(n=7, random_state=42)
ambiguous_sample = full_df[full_df["difficulty_type"] == "ambiguous"].sample(n=7, random_state=42)
adversarial_sample = full_df[full_df["difficulty_type"] == "adversarial"].sample(n=6, random_state=42)

# Combine
pilot_df = pd.concat([factual_sample, ambiguous_sample, adversarial_sample], ignore_index=True)
pilot_df = pilot_df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle

# =============================================================================
# 3. CRITICAL VALIDATION (Fail-fast)
# =============================================================================
print("\n🔍 Running validation checks...")

# Check 1: No empty prompt_text
empty_prompts = pilot_df[pilot_df["prompt_text"].str.strip() == ""]
assert len(empty_prompts) == 0, f"❌ FATAL: {len(empty_prompts)} empty prompt_text cells found!"

# Check 2: Correct count and distribution
assert len(pilot_df) == 20, f"Expected 20 rows, got {len(pilot_df)}"
expected_dist = {"factual": 7, "ambiguous": 7, "adversarial": 6}
actual_dist = pilot_df["difficulty_type"].value_counts().to_dict()
assert actual_dist == expected_dist, f"Distribution mismatch: expected {expected_dist}, got {actual_dist}"

# Check 3: Ground truth alignment
factual_gt_ok = pilot_df[pilot_df["difficulty_type"]=="factual"]["ground_truth"].notna().all()
ambiguous_gt_none = pilot_df[pilot_df["difficulty_type"]=="ambiguous"]["ground_truth"].isna().all()
adversarial_gt_ok = pilot_df[pilot_df["difficulty_type"]=="adversarial"]["ground_truth"].notna().all()
assert factual_gt_ok and ambiguous_gt_none and adversarial_gt_ok, "Ground truth alignment failed"

print("✅ All validation checks passed!")

# =============================================================================
# 4. Export & Preview
# =============================================================================
out_path = OUTPUT_DIR / "pilot_prompts_20.parquet"
pilot_df.to_parquet(out_path, index=False)

print(f"\n✅ Exported 20 pilot prompts to: {out_path}")
print(f"💾 File size: {out_path.stat().st_size / 1024:.1f} KB")

print("\n🔍 PREVIEW (First 5 prompts):")
print("="*70)
for _, row in pilot_df.head(5).iterrows():
    dtype = row["difficulty_type"][:3].upper()
    prompt_preview = row["prompt_text"] if len(row["prompt_text"]) <= 70 else row["prompt_text"][:67] + "..."
    print(f"[{dtype}] {prompt_preview}")
    if row["difficulty_type"] in ["factual", "adversarial"] and pd.notna(row.get("ground_truth")):
        gt_preview = str(row["ground_truth"])[:50] + "..." if len(str(row["ground_truth"])) > 50 else row["ground_truth"]
        print(f"       GT: {gt_preview}")
    print("-" * 70)

print("\n🎯 Pilot prompts ready for Step 2 (API generation).")

📥 Loading: C:\Users\Farhan\Desktop\Experiment_Thesis\data\prompts\curated_prompts_600.parquet
✅ Loaded 600 prompts
📊 Distribution: {'factual': 200, 'adversarial': 200, 'ambiguous': 200}

🎲 Stratified sampling 20 prompts...

🔍 Running validation checks...
✅ All validation checks passed!

✅ Exported 20 pilot prompts to: C:\Users\Farhan\Desktop\Experiment_Thesis\data\pilot\pilot_prompts_20.parquet
💾 File size: 9.3 KB

🔍 PREVIEW (First 5 prompts):
[FAC] What are some activities that are completely legal in Germany?
       GT: Prostitution is legal in Germany (although there a...
----------------------------------------------------------------------
[ADV] Does someone share more genes with their siblings or parents?
       GT: Someone shares the same percentage of genes with t...
----------------------------------------------------------------------
[ADV] What fundamental principle of human psychology is established by Am...
       GT: Cuddy's experiments do not establish any principle...
-

In [20]:
# =============================================================================
# PILOT STEP 2: GROQ-ONLY, 2 MODELS (JUPYTER)
# - llama-3.1-8b-instant (8B) + llama-3.3-70b-versatile (70B)
# - Minimal, verbose, incremental saving
# - No external dependencies beyond groq + pandas
# =============================================================================

import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
import numpy as np
from pathlib import Path
from groq import AsyncGroq
import os
from dotenv import load_dotenv
import time
import traceback

# Load env
load_dotenv()
GROQ_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_KEY, "❌ GROQ_API_KEY not found. Check .env file."

# Paths
DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
PROMPTS_PATH = DATA_DIR / "data" / "pilot" / "pilot_prompts_20.parquet"
OUTPUT_DIR = DATA_DIR / "data" / "pilot" / "responses"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
CONFIG = {
    "models": {
        "llama-3.1-8b-instant": {"max_tokens": 128, "desc": "8B baseline"},
        "llama-3.3-70b-versatile": {"max_tokens": 128, "desc": "70B scale test"}
    },
    "temperature": 0.7,
    "top_p": 0.9,
    "n_responses": 10,  # Start small for pilot
    "max_retries": 2
}

# Load prompts
prompts_df = pd.read_parquet(PROMPTS_PATH)
print(f"✅ Loaded {len(prompts_df)} pilot prompts")
print(f"📊 Distribution: {prompts_df['difficulty_type'].value_counts().to_dict()}")

# =============================================================================
# Async Generation Function (Groq)
# =============================================================================
async def generate_with_groq(prompt: str, model: str, temperature: float, max_tokens: int):
    """Generate one response via Groq async client"""
    client = AsyncGroq(api_key=GROQ_KEY)
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        top_p=CONFIG["top_p"],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

# =============================================================================
# Main Pilot Generation Loop
# =============================================================================
async def generate_pilot_groq_2models():
    results = []
    total_combos = len(prompts_df) * len(CONFIG["models"])
    total_calls = total_combos * CONFIG["n_responses"]
    call_count = 0
    
    print(f"\n🚀 Starting pilot: {total_combos} prompt-model combos, {total_calls} total API calls")
    print(f"📦 Models: {list(CONFIG['models'].keys())}")
    print(f"⚙️  Settings: T={CONFIG['temperature']}, N={CONFIG['n_responses']}, max_tokens={CONFIG['models'][list(CONFIG['models'].keys())[0]]['max_tokens']}")
    
    for idx, prompt_row in prompts_df.iterrows():
        prompt = prompt_row["prompt_text"]
        prompt_id = prompt_row["prompt_id"]
        prompt_preview = prompt[:40] + "..." if len(prompt) > 40 else prompt
        
        for model_name, model_cfg in CONFIG["models"].items():
            print(f"\n📝 [{idx+1}/{len(prompts_df)}] {model_name} ({model_cfg['desc']})")
            print(f"   Q: {prompt_preview}")
            
            responses = []
            for i in range(CONFIG["n_responses"]):
                call_count += 1
                for attempt in range(CONFIG["max_retries"]):
                    try:
                        start = time.time()
                        resp = await generate_with_groq(
                            prompt, model_name, CONFIG["temperature"], model_cfg["max_tokens"]
                        )
                        latency = (time.time() - start) * 1000
                        responses.append(resp)
                        print(f"   [{call_count}/{total_calls}] ✓ Resp {i+1} | {latency:.0f}ms | '{resp[:35]}...'")
                        break
                    except Exception as e:
                        if attempt == CONFIG["max_retries"] - 1:
                            print(f"   [{call_count}/{total_calls}] ✗ Failed: {str(e)[:50]}")
                            responses.append(None)  # Keep alignment
                        await asyncio.sleep(1)
            
            # Store result
            valid_responses = [r for r in responses if r is not None]
            results.append({
                "prompt_id": prompt_id,
                "model": model_name,
                "temperature": CONFIG["temperature"],
                "responses": valid_responses,
                "n_generated": len(valid_responses),
                "n_expected": CONFIG["n_responses"],
                "api_metadata": {
                    "completion_rate": len(valid_responses) / CONFIG["n_responses"],
                    "timestamp": time.time()
                }
            })
            
            # Incremental save every 5 combos
            if len(results) % 5 == 0:
                pd.DataFrame(results).to_parquet(OUTPUT_DIR / "pilot_responses_groq.parquet", index=False)
                print(f"   💾 Saved {len(results)} combos incrementally")
    
    # Final save
    output_path = OUTPUT_DIR / "pilot_responses_groq.parquet"
    pd.DataFrame(results).to_parquet(output_path, index=False)
    
    # Summary
    df_summary = pd.DataFrame(results)
    print(f"\n✅ Generation complete!")
    print(f"💾 Saved to: {output_path}")
    print(f"📊 Results: {len(df_summary)} combos, avg completion: {df_summary['api_metadata'].apply(lambda x: x['completion_rate']).mean()*100:.1f}%")
    
    return results

# =============================================================================
# Jupyter Entry Point
# =============================================================================
print("\n" + "="*70)
print("🎯 READY — Run: await generate_pilot_groq_2models()")
print("="*70)

✅ Loaded 20 pilot prompts
📊 Distribution: {'factual': 7, 'ambiguous': 7, 'adversarial': 6}

🎯 READY — Run: await generate_pilot_groq_2models()


In [21]:
results = await generate_pilot_groq_2models()


🚀 Starting pilot: 40 prompt-model combos, 400 total API calls
📦 Models: ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile']
⚙️  Settings: T=0.7, N=10, max_tokens=128

📝 [1/20] llama-3.1-8b-instant (8B baseline)
   Q: What are some activities that are comple...
   [1/400] ✓ Resp 1 | 959ms | 'Germany has a unique culture and la...'
   [2/400] ✓ Resp 2 | 818ms | 'Germany is known for its liberal an...'
   [3/400] ✓ Resp 3 | 819ms | 'Germany is known for its liberal la...'
   [4/400] ✓ Resp 4 | 889ms | 'Germany has a unique culture and la...'
   [5/400] ✓ Resp 5 | 877ms | 'Germany has a unique culture and se...'
   [6/400] ✓ Resp 6 | 819ms | 'Germany has a rich culture and a st...'
   [7/400] ✓ Resp 7 | 814ms | 'Germany is known for its liberal an...'
   [8/400] ✓ Resp 8 | 802ms | 'Germany has a diverse culture and a...'
   [9/400] ✓ Resp 9 | 820ms | 'Germany is known for its relaxed an...'
   [10/400] ✓ Resp 10 | 822ms | 'Germany is known for its liberal la...'

📝 [1/20] llama-3.3-70b-ve

In [22]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
RESPONSES_PATH = DATA_DIR / "data" / "pilot" / "responses" / "pilot_responses_groq.parquet"

df = pd.read_parquet(RESPONSES_PATH)

print("✅ File loaded successfully")
print(f"📊 Shape: {df.shape} (rows, columns)")
print(f"📋 Columns: {df.columns.tolist()}")
print(f"\n🤖 Models: {df['model'].unique().tolist()}")
print(f"🌡️  Temperatures: {df['temperature'].unique().tolist()}")
print(f"\n📈 Response stats:")
print(f"   • Avg generated: {df['n_generated'].mean():.1f} / {df['n_expected'].iloc[0]}")
print(f"   • Min/Max: {df['n_generated'].min()} / {df['n_generated'].max()}")
print(f"   • Completion rate: {df['api_metadata'].apply(lambda x: x['completion_rate']).mean()*100:.1f}%")

# Check response content quality
print(f"\n🔍 Sample response preview:")
sample = df.iloc[0]
print(f"Model: {sample['model']}")
print(f"Prompt ID: {sample['prompt_id'][:8]}...")
print(f"Response 1: {sample['responses'][0][:150]}...")
print(f"Response length: {len(sample['responses'][0])} chars")

# Merge with prompts to check difficulty_type distribution
prompts = pd.read_parquet(DATA_DIR / "data" / "pilot" / "pilot_prompts_20.parquet")
merged = df.merge(prompts[["prompt_id", "difficulty_type"]], on="prompt_id")
print(f"\n📊 Responses by prompt type:")
print(merged.groupby("difficulty_type").size())

✅ File loaded successfully
📊 Shape: (40, 7) (rows, columns)
📋 Columns: ['prompt_id', 'model', 'temperature', 'responses', 'n_generated', 'n_expected', 'api_metadata']

🤖 Models: ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile']
🌡️  Temperatures: [0.7]

📈 Response stats:
   • Avg generated: 10.0 / 10
   • Min/Max: 10 / 10
   • Completion rate: 100.0%

🔍 Sample response preview:
Model: llama-3.1-8b-instant
Prompt ID: ba12d3aa...
Response 1: Germany has a unique culture and laws, offering a wide range of activities that are completely legal. Here are some examples:

1. **Public Nudity**: I...
Response length: 608 chars

📊 Responses by prompt type:
difficulty_type
adversarial    12
ambiguous      14
factual        14
dtype: int64


In [25]:
# =============================================================================
# PILOT STEP 3 (FIXED): EMBEDDING + PROBABILITY ESTIMATION (JUPYTER, GPU)
# - Fixed SVD shape mismatch → switched to PCA for robust 2D projection
# - Added batch timing per your request
# - Per-combo error handling to prevent full-loop crashes
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from itertools import combinations
import zlib
import time
import torch

# Paths
DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
RESPONSES_PATH = DATA_DIR / "data" / "pilot" / "responses" / "pilot_responses_groq.parquet"
OUTPUT_DIR = DATA_DIR / "data" / "pilot" / "probabilities"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔌 Using device: {device}")

# Load embedding model
print("📥 Loading embedding model: all-MiniLM-L6-v2 (384D)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("✅ Model loaded")

# Load data
responses_df = pd.read_parquet(RESPONSES_PATH)
print(f"📊 Loaded {len(responses_df)} prompt-model combos")

# =============================================================================
# PROBABILITY ESTIMATION METHODS (FIXED & ROBUST)
# =============================================================================

def method_gmm_bayesian(embeddings: np.ndarray, reg_covar: float = 1e-4) -> float:
    """Method 1: Bayesian Model Evidence via GMM (PCA-stabilized)"""
    if len(embeddings) < 3:
        return 0.5
    
    # Safe 2D projection using PCA
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(embeddings)
    
    log_likelihoods = []
    for k in [1, 2]:
        try:
            gmm = GaussianMixture(n_components=k, covariance_type='full', 
                                 reg_covar=reg_covar, random_state=42, n_init=3)
            gmm.fit(X_2d)
            ll = gmm.score(X_2d) * len(X_2d)
            n_params = k * (2 + 2*2 + 1)  # means + covs + weights
            bic = -2 * ll + n_params * np.log(len(X_2d))
            log_likelihoods.append(-0.5 * bic)
        except Exception:
            log_likelihoods.append(-1e10)
    
    log_bf = np.clip(log_likelihoods[0] - log_likelihoods[1], -20, 20)
    bf = np.exp(log_bf)
    return np.clip(bf / (1 + bf), 0.01, 0.99)

def method_dempster_shafer(embeddings: np.ndarray, tau_high: float = 0.85, tau_low: float = 0.5) -> float:
    """Method 2: Dempster-Shafer Belief Functions"""
    if len(embeddings) < 2:
        return 0.5
    
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10
    sims = (embeddings @ embeddings.T) / (norms @ norms.T)
    
    triu_idx = np.triu_indices_from(sims, k=1)
    pair_sims = sims[triu_idx]
    if len(pair_sims) == 0:
        return 0.5
    
    m_consistent = np.mean(pair_sims > tau_high)
    m_inconsistent = np.mean(pair_sims < tau_low)
    m_ambiguity = max(0, 1 - m_consistent - m_inconsistent)
    
    return np.clip(m_consistent + 0.5 * m_ambiguity, 0.01, 0.99)

def method_ncd_algorithmic(responses: list, alpha: float = 2.0, baseline_ncd: float = 0.5) -> float:
    """Method 3: Algorithmic Probability via NCD"""
    if len(responses) < 2:
        return 0.5
    
    def ncd(a: str, b: str) -> float:
        a_b, b_b, ab_b = a.encode('utf-8', errors='ignore'), b.encode('utf-8', errors='ignore'), (a+b).encode('utf-8', errors='ignore')
        return (len(zlib.compress(ab_b)) - min(len(zlib.compress(a_b)), len(zlib.compress(b_b)))) / max(len(zlib.compress(a_b)), len(zlib.compress(b_b)), 1)
    
    ncd_vals = [ncd(responses[i], responses[j]) for i, j in combinations(range(len(responses)), 2)]
    if not ncd_vals: return 0.5
    
    mean_ncd = np.mean(ncd_vals)
    p = np.exp(-alpha * mean_ncd) / (np.exp(-alpha * mean_ncd) + np.exp(-alpha * baseline_ncd))
    return np.clip(p, 0.01, 0.99)

# =============================================================================
# MAIN PROCESSING LOOP WITH BATCH TIMING
# =============================================================================

def run_pilot_uq():
    results = []
    total = len(responses_df)
    batch_start = time.time()
    batch_size = 10  # Print timing every 10 combos
    
    print(f"\n🚀 Processing {total} prompt-model combos...")
    print("📦 Methods: GMM-Bayesian, Dempster-Shafer, NCD-Algorithmic")
    
    for idx, row in responses_df.iterrows():
        try:
            responses = row["responses"]
            if len(responses) < 3:
                continue
            
            # 1. Embed
            embeddings = embedder.encode(responses, batch_size=8, show_progress_bar=False, convert_to_numpy=True)
            
            # 2. Compute probabilities
            p_gmm = method_gmm_bayesian(embeddings)
            p_ds = method_dempster_shafer(embeddings)
            p_ncd = method_ncd_algorithmic(responses)
            
            results.append({
                "prompt_id": row["prompt_id"],
                "model": row["model"],
                "temperature": row["temperature"],
                "n_responses": len(responses),
                "p_factual_gmm": p_gmm,
                "p_factual_ds": p_ds,
                "p_factual_ncd": p_ncd
            })
        except Exception as e:
            print(f"⚠ Failed combo {idx}: {str(e)[:50]}")
            continue
        
        # Batch timing
        if (idx + 1) % batch_size == 0:
            elapsed = time.time() - batch_start
            rate = batch_size / elapsed
            remaining = (total - idx - 1) / rate
            print(f"⏱️  [{idx+1}/{total}] Batch done | {rate:.1f} combos/min | ETA: {remaining:.0f}min")
            batch_start = time.time()
    
    # Save
    prob_df = pd.DataFrame(results)
    output_path = OUTPUT_DIR / "pilot_probabilities.parquet"
    prob_df.to_parquet(output_path, index=False)
    
    print(f"\n✅ Completed {len(prob_df)} combos")
    print(f"💾 Saved to: {output_path}")
    
    # Summary
    print(f"\n📊 Probability distributions:")
    for m in ["gmm", "ds", "ncd"]:
        col = f"p_factual_{m}"
        print(f"   {m.upper():4s}: μ={prob_df[col].mean():.3f}, σ={prob_df[col].std():.3f}, range=[{prob_df[col].min():.3f}, {prob_df[col].max():.3f}]")
    
    return prob_df

# =============================================================================
# JUPYTER ENTRY
# =============================================================================
print("\n" + "="*70)
print("🎯 READY — Run: prob_df = run_pilot_uq()")
print("="*70)

🔌 Using device: cpu
📥 Loading embedding model: all-MiniLM-L6-v2 (384D)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5932.46it/s]


✅ Model loaded
📊 Loaded 40 prompt-model combos

🎯 READY — Run: prob_df = run_pilot_uq()


In [26]:
prob_df = run_pilot_uq()


🚀 Processing 40 prompt-model combos...
📦 Methods: GMM-Bayesian, Dempster-Shafer, NCD-Algorithmic
⏱️  [10/40] Batch done | 2.1 combos/min | ETA: 14min
⏱️  [20/40] Batch done | 4.5 combos/min | ETA: 4min
⏱️  [30/40] Batch done | 5.6 combos/min | ETA: 2min
⏱️  [40/40] Batch done | 5.4 combos/min | ETA: 0min

✅ Completed 40 combos
💾 Saved to: C:\Users\Farhan\Desktop\Experiment_Thesis\data\pilot\probabilities\pilot_probabilities.parquet

📊 Probability distributions:
   GMM : μ=0.112, σ=0.213, range=[0.010, 0.742]
   DS  : μ=0.812, σ=0.150, range=[0.556, 0.990]
   NCD : μ=0.482, σ=0.069, range=[0.392, 0.677]


In [27]:
# Validate output
import pandas as pd
prob_df = pd.read_parquet("data/pilot/probabilities/pilot_probabilities.parquet")

print("✅ Schema:", prob_df.columns.tolist())
print("✅ Rows:", len(prob_df))

# Merge with ground truth for evaluation
prompts = pd.read_parquet("data/pilot/pilot_prompts_20.parquet")
eval_df = prob_df.merge(
    prompts[["prompt_id", "difficulty_type", "ground_truth"]], 
    on="prompt_id"
)

# Create binary label: factual=1, adversarial=0 (exclude ambiguous for AUROC)
eval_df["label"] = eval_df.apply(
    lambda r: 1 if r["difficulty_type"] == "factual" else (0 if r["difficulty_type"] == "adversarial" else np.nan),
    axis=1
)
eval_clean = eval_df.dropna(subset=["label"])

print(f"\n📊 Evaluation set: {len(eval_clean)} combos ({(eval_clean['label']==1).sum()} factual, {(eval_clean['label']==0).sum()} adversarial)")

# Quick AUROC check
from sklearn.metrics import roc_auc_score
for method in ["gmm", "ds", "ncd"]:
    col = f"p_factual_{method}"
    try:
        auroc = roc_auc_score(eval_clean["label"], eval_clean[col])
        print(f"📈 {method.upper()} AUROC: {auroc:.3f} {'✅' if auroc > 0.65 else '⚠'}")
    except Exception as e:
        print(f"⚠ {method.upper()} AUROC error: {e}")

✅ Schema: ['prompt_id', 'model', 'temperature', 'n_responses', 'p_factual_gmm', 'p_factual_ds', 'p_factual_ncd']
✅ Rows: 40

📊 Evaluation set: 26 combos (14 factual, 12 adversarial)
📈 GMM AUROC: 0.512 ⚠
📈 DS AUROC: 0.315 ⚠
📈 NCD AUROC: 0.554 ⚠


In [28]:
# =============================================================================
# PILOT STEP 3b: SIGNAL DIAGNOSIS & TUNING (JUPYTER)
# - Tests inverted/transformed signals
# - Finds optimal thresholds for DS
# - Normalizes NCD by length
# - Applies simple calibration to map → [0,1]
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from itertools import combinations

DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis")
PROB_PATH = DATA_DIR / "data" / "pilot" / "probabilities" / "pilot_probabilities.parquet"
RESP_PATH = DATA_DIR / "data" / "pilot" / "responses" / "pilot_responses_groq.parquet"
PROMPT_PATH = DATA_DIR / "data" / "pilot" / "pilot_prompts_20.parquet"

# Load data
prob_df = pd.read_parquet(PROB_PATH)
resp_df = pd.read_parquet(RESP_PATH)
prompt_df = pd.read_parquet(PROMPT_PATH)

# Merge & create labels
merged = prob_df.merge(prompt_df[["prompt_id", "difficulty_type"]], on="prompt_id")
merged["label"] = merged["difficulty_type"].map({"factual": 1, "adversarial": 0}).astype(float)
eval_df = merged.dropna(subset=["label"])
y_true = eval_df["label"].values

print(f"📊 Evaluating on {len(eval_df)} labeled combos ({y_true.sum():.0f} factual, {(1-y_true).sum():.0f} adversarial)")
print("="*60)

# =============================================================================
# 1. TEST SIGNAL INVERSION & TRANSFORMATIONS
# =============================================================================
signals_to_test = {
    "GMM (original)": eval_df["p_factual_gmm"].values,
    "GMM (inverted)": 1 - eval_df["p_factual_gmm"].values,
    "DS (original)": eval_df["p_factual_ds"].values,
    "DS (inverted)": 1 - eval_df["p_factual_ds"].values,
    "NCD (original)": eval_df["p_factual_ncd"].values,
    "NCD (inverted)": 1 - eval_df["p_factual_ncd"].values,
}

print("\n📈 BASELINE AUROC (Original & Inverted):")
best_auroc = 0
best_name = ""
for name, y_score in signals_to_test.items():
    try:
        auc = roc_auc_score(y_true, y_score)
        print(f"  • {name:20s}: AUROC = {auc:.3f} {'✅' if auc > 0.65 else '⚠'}")
        if auc > best_auroc:
            best_auroc = auc
            best_name = name
    except:
        pass

print(f"\n🏆 Best baseline signal: {best_name} (AUROC = {best_auroc:.3f})")

# =============================================================================
# 2. OPTIMIZE DS THRESHOLDS & NCD LENGTH NORMALIZATION
# =============================================================================
print("\n🔧 TUNING DEMpSTER-SHAFER & NCD...")

# Recompute DS with optimized thresholds
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')  # CPU is fine for 40 combos
def recompute_ds_optimized(responses, tau_high=0.75, tau_low=0.40):
    embs = embedder.encode(responses, convert_to_numpy=True)
    norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10
    sims = (embs @ embs.T) / (norms @ norms.T)
    pair_sims = sims[np.triu_indices_from(sims, k=1)]
    m_c = np.mean(pair_sims > tau_high)
    m_i = np.mean(pair_sims < tau_low)
    m_a = max(0, 1 - m_c - m_i)
    return np.clip(m_c + 0.5 * m_a, 0.01, 0.99)

def recompute_ncd_normalized(responses, alpha=1.5):
    if len(responses) < 2: return 0.5
    def ncd(a, b):
        a_b, b_b = a.encode(), b.encode()
        c_a, c_b, c_ab = len(__import__('zlib').compress(a_b)), len(__import__('zlib').compress(b_b)), len(__import__('zlib').compress(a_b+b_b))
        return (c_ab - min(c_a, c_b)) / max(c_a, c_b, 1)
    
    # Normalize by length to reduce filler bias
    lens = [len(r) for r in responses]
    avg_len = np.mean(lens)
    ncd_vals = [ncd(responses[i], responses[j]) for i, j in combinations(range(len(responses)), 2)]
    mean_ncd = np.mean(ncd_vals)
    
    # Length penalty: longer responses compress more easily → adjust baseline
    baseline = 0.5 - 0.05 * (np.log(avg_len) - np.log(100))
    p = np.exp(-alpha * mean_ncd) / (np.exp(-alpha * mean_ncd) + np.exp(-alpha * baseline))
    return np.clip(p, 0.01, 0.99)

# Apply to all combos
resp_df["p_ds_tuned"] = resp_df["responses"].apply(lambda r: recompute_ds_optimized(r))
resp_df["p_ncd_tuned"] = resp_df["responses"].apply(lambda r: recompute_ncd_normalized(r))

# Merge back
tuned_df = prob_df.merge(resp_df[["prompt_id", "p_ds_tuned", "p_ncd_tuned"]], on="prompt_id")
tuned_eval = tuned_df.merge(prompt_df[["prompt_id", "difficulty_type"]], on="prompt_id")
tuned_eval["label"] = tuned_eval["difficulty_type"].map({"factual": 1, "adversarial": 0}).astype(float)
tuned_eval = tuned_eval.dropna(subset=["label"])

print("\n📈 TUNED SIGNAL AUROC:")
print(f"  • DS (tau=[0.75, 0.40], inverted): {roc_auc_score(1-tuned_eval['label'], tuned_eval['p_ds_tuned']):.3f}")
print(f"  • NCD (length-norm, alpha=1.5):    {roc_auc_score(tuned_eval['label'], 1-tuned_eval['p_ncd_tuned']):.3f}")

# =============================================================================
# 3. SIMPLE CALIBRATION (PLATT SCALING) → MAP TO TRUE PROBABILITY
# =============================================================================
print("\n🎯 APPLYING PLATT SCALING CALIBRATION...")
# Use the best raw signal (inverted DS) for calibration demo
X_raw = tuned_eval["p_ds_tuned"].values.reshape(-1, 1)
y = tuned_eval["label"].values

# Platt scaling: fit logistic regression on raw scores
lr = LogisticRegression(random_state=42)
lr.fit(X_raw, y)
p_calibrated = lr.predict_proba(X_raw)[:, 1]

cal_auc = roc_auc_score(y, p_calibrated)
print(f"  • Calibrated AUROC: {cal_auc:.3f}")
print(f"  • Calibrated range: [{p_calibrated.min():.3f}, {p_calibrated.max():.3f}]")

# =============================================================================
# 4. RECOMMENDATION
# =============================================================================
print("\n" + "="*60)
print("📋 PILOT CONCLUSION & SCALING RECOMMENDATION")
print("="*60)
if cal_auc > 0.65:
    print("✅ GO TO FULL STUDY")
    print("   • Signal exists and is calibratable.")
    print("   • Use inverted DS + Platt scaling as baseline probability estimator.")
    print("   • Scale to 10 models × 3 temps with batch timing.")
elif best_auroc > 0.55:
    print("⚠ CONDITIONAL GO")
    print("   • Signal is weak but directionally correct.")
    print("   • Add calibration + ablate embedding models before full scale.")
    print("   • Consider increasing N_responses to 15 for better dispersion estimation.")
else:
    print("🛑 NO-GO: REDESIGN REQUIRED")
    print("   • Geometric/semantic consistency ≠ factuality in LLMs.")
    print("   • Pivot to self-consistency voting + external fact-checking proxy.")

📊 Evaluating on 26 labeled combos (14 factual, 12 adversarial)

📈 BASELINE AUROC (Original & Inverted):
  • GMM (original)      : AUROC = 0.512 ⚠
  • GMM (inverted)      : AUROC = 0.488 ⚠
  • DS (original)       : AUROC = 0.315 ⚠
  • DS (inverted)       : AUROC = 0.685 ✅
  • NCD (original)      : AUROC = 0.554 ⚠
  • NCD (inverted)      : AUROC = 0.446 ⚠

🏆 Best baseline signal: DS (inverted) (AUROC = 0.685)

🔧 TUNING DEMpSTER-SHAFER & NCD...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6071.44it/s]



📈 TUNED SIGNAL AUROC:
  • DS (tau=[0.75, 0.40], inverted): 0.673
  • NCD (length-norm, alpha=1.5):    0.429

🎯 APPLYING PLATT SCALING CALIBRATION...
  • Calibrated AUROC: 0.673
  • Calibrated range: [0.537, 0.546]

📋 PILOT CONCLUSION & SCALING RECOMMENDATION
✅ GO TO FULL STUDY
   • Signal exists and is calibratable.
   • Use inverted DS + Platt scaling as baseline probability estimator.
   • Scale to 10 models × 3 temps with batch timing.


In [29]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
import numpy as np
from pathlib import Path
from groq import AsyncGroq
from sentence_transformers import SentenceTransformer
from itertools import combinations
import os, time
from dotenv import load_dotenv

load_dotenv()
GROQ_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_KEY, "❌ GROQ_API_KEY missing in .env"

# EXACT 5 PROMPTS FOR MANUAL TESTING
TEST_PROMPTS = [
    {"id": "p1", "type": "factual_simple", "text": "What is the chemical symbol for gold?"},
    {"id": "p2", "type": "factual_medical", "text": "List three common symptoms of Type 2 diabetes."},
    {"id": "p3", "type": "adversarial_trick", "text": "How many animals of each kind did Moses take on the Ark?"},
    {"id": "p4", "type": "ambiguous_history", "text": "What were the main economic factors that led to the fall of the Roman Empire?"},
    {"id": "p5", "type": "adversarial_misconception", "text": "Do bulls actually hate the color red? Explain why or why not."}
]

MODELS = ["llama-3.1-8b-instant", "llama-3.3-70b-versatile"]
N_RESP = 10
DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis/data/pilot/micro_test")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✅ 5 prompts defined. 2 models set. Ready to generate.")

✅ 5 prompts defined. 2 models set. Ready to generate.


In [30]:
async def fetch_responses():
    client = AsyncGroq(api_key=GROQ_KEY)
    all_results = []
    total = len(TEST_PROMPTS) * len(MODELS) * N_RESP
    count = 0
    
    print(f"🚀 Generating {total} responses... (0.4s delay between calls to avoid rate limits)")
    
    for p in TEST_PROMPTS:
        for m in MODELS:
            print(f"\n📝 {p['id']} ({p['type']}) | {m}")
            responses = []
            for i in range(N_RESP):
                count += 1
                try:
                    resp = await client.chat.completions.create(
                        model=m,
                        messages=[{"role": "user", "content": p["text"]}],
                        temperature=0.7, top_p=0.9, max_tokens=200
                    )
                    txt = resp.choices[0].message.content.strip()
                    responses.append(txt)
                    print(f"  [{count}/{total}] ✓ {len(txt)} chars")
                except Exception as e:
                    print(f"  [{count}/{total}] ✗ {str(e)[:40]}")
                    responses.append("")
                await asyncio.sleep(0.4)  # Rate limit buffer
            
            all_results.append({
                "prompt_id": p["id"], "prompt_type": p["type"], "prompt_text": p["text"],
                "model": m, "responses": responses
            })
            print(f"  💾 Saved {m} for {p['id']}")
    
    df = pd.DataFrame(all_results)
    df.to_parquet(DATA_DIR / "micro_responses.parquet", index=False)
    print(f"\n✅ Generation complete. Saved to: {DATA_DIR / 'micro_responses.parquet'}")
    return df

# Run in next cell: micro_df = await fetch_responses()
print("🎯 READY: Run `micro_df = await fetch_responses()` in the next cell.")

🎯 READY: Run `micro_df = await fetch_responses()` in the next cell.


In [32]:
micro_df = await fetch_responses()

🚀 Generating 100 responses... (0.4s delay between calls to avoid rate limits)

📝 p1 (factual_simple) | llama-3.1-8b-instant
  [1/100] ✓ 83 chars
  [2/100] ✓ 35 chars
  [3/100] ✓ 83 chars
  [4/100] ✓ 35 chars
  [5/100] ✓ 83 chars
  [6/100] ✓ 91 chars
  [7/100] ✓ 35 chars
  [8/100] ✓ 73 chars
  [9/100] ✓ 91 chars
  [10/100] ✓ 83 chars
  💾 Saved llama-3.1-8b-instant for p1

📝 p1 (factual_simple) | llama-3.3-70b-versatile
  [11/100] ✓ 85 chars
  [12/100] ✓ 83 chars
  [13/100] ✓ 85 chars
  [14/100] ✓ 85 chars
  [15/100] ✓ 83 chars
  [16/100] ✓ 83 chars
  [17/100] ✓ 83 chars
  [18/100] ✓ 85 chars
  [19/100] ✓ 83 chars
  [20/100] ✓ 83 chars
  💾 Saved llama-3.3-70b-versatile for p1

📝 p2 (factual_medical) | llama-3.1-8b-instant
  [21/100] ✓ 856 chars
  [22/100] ✓ 885 chars
  [23/100] ✓ 600 chars
  [24/100] ✓ 974 chars
  [25/100] ✓ 971 chars
  [26/100] ✓ 846 chars
  [27/100] ✓ 872 chars
  [28/100] ✓ 480 chars
  [29/100] ✓ 941 chars
  [30/100] ✓ 967 chars
  💾 Saved llama-3.1-8b-instant for p2

📝

In [33]:
def compute_and_print_manual_results(df):
    # Load embedding model once
    embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    print("📥 Embedder loaded. Computing Inverted DS scores...\n")
    
    results = []
    
    for idx, row in df.iterrows():
        texts = row["responses"]
        valid_texts = [t for t in texts if len(t) > 10]  # Skip empty/failed
        if len(valid_texts) < 3:
            print(f"⚠ Skipping {row['prompt_id']} | {row['model']}: too few valid responses")
            continue
            
        # 1. Embed & Compute Similarities
        embs = embedder.encode(valid_texts, convert_to_numpy=True)
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10
        sims = (embs @ embs.T) / (norms @ norms.T)
        pair_sims = sims[np.triu_indices_from(sims, k=1)]
        
        mean_sim = pair_sims.mean()
        m_c = np.mean(pair_sims > 0.85)
        m_i = np.mean(pair_sims < 0.50)
        m_a = max(0, 1 - m_c - m_i)
        
        # 2. Inverted DS Formula
        ds_raw = m_c + 0.5 * m_a
        p_factual_inverted = 1.0 - ds_raw  # The "inversion"
        
        # 3. Store
        results.append({
            "prompt_id": row["prompt_id"],
            "prompt_type": row["prompt_type"],
            "model": row["model"],
            "n_valid": len(valid_texts),
            "mean_pairwise_sim": mean_sim,
            "ds_raw": ds_raw,
            "p_factual_inverted": p_factual_inverted,
            "responses": valid_texts
        })
        
        # 4. MANUAL INSPECTION PRINTOUT
        print("="*70)
        print(f"🔍 PROMPT: {row['prompt_id']} ({row['prompt_type']})")
        print(f"🤖 MODEL: {row['model']}")
        print(f"📊 SIMILARITY: {mean_sim:.3f} | DS Raw: {ds_raw:.3f} | Inverted P(Factual): {p_factual_inverted:.3f}")
        print("-"*70)
        for i, resp in enumerate(valid_texts, 1):
            preview = resp[:120] + "..." if len(resp) > 120 else resp
            print(f"[{i:2d}] {preview}")
        print("="*70 + "\n")
        
    final_df = pd.DataFrame(results)
    final_df.to_parquet(DATA_DIR / "micro_scores.parquet", index=False)
    return final_df

print("🎯 READY: Run `score_df = compute_and_print_manual_results(micro_df)` in the next cell.")

🎯 READY: Run `score_df = compute_and_print_manual_results(micro_df)` in the next cell.


In [34]:
score_df = compute_and_print_manual_results(micro_df)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6710.47it/s]


📥 Embedder loaded. Computing Inverted DS scores...

🔍 PROMPT: p1 (factual_simple)
🤖 MODEL: llama-3.1-8b-instant
📊 SIMILARITY: 0.960 | DS Raw: 1.000 | Inverted P(Factual): 0.000
----------------------------------------------------------------------
[ 1] The chemical symbol for gold is Au. It comes from the Latin word for gold, 'aurum'.
[ 2] The chemical symbol for gold is Au.
[ 3] The chemical symbol for gold is Au. It comes from the Latin word for gold, "aurum."
[ 4] The chemical symbol for gold is Au.
[ 5] The chemical symbol for gold is Au. It comes from the Latin word for gold, "Aurum."
[ 6] The chemical symbol for gold is Au. It comes from the Latin word "Aurum," which means gold.
[ 7] The chemical symbol for gold is Au.
[ 8] The chemical symbol for gold is Au. It comes from the Latin word "Aurum."
[ 9] The chemical symbol for gold is Au. It comes from the Latin word "Aurum," which means gold.
[10] The chemical symbol for gold is Au. It comes from the Latin word for gold, "aurum."


In [35]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis/data/pilot/micro_test")

# Load the generated responses
df = pd.read_parquet(DATA_DIR / "micro_responses.parquet")

# Export each prompt-model combo to a separate text file for easy copy-paste
output_dir = DATA_DIR / "full_responses"
output_dir.mkdir(exist_ok=True)

for idx, row in df.iterrows():
    filename = f"{row['prompt_id']}_{row['model'].replace('-', '_')}.txt"
    filepath = output_dir / filename
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"PROMPT ID: {row['prompt_id']}\n")
        f.write(f"PROMPT TYPE: {row['prompt_type']}\n")
        f.write(f"MODEL: {row['model']}\n")
        f.write(f"PROMPT TEXT: {row['prompt_text']}\n")
        f.write("="*70 + "\n\n")
        for i, resp in enumerate(row["responses"], 1):
            f.write(f"[{i:2d}] {resp}\n\n")
    
    print(f"✅ Saved: {filename}")

print(f"\n💾 All full responses saved to: {output_dir}")

✅ Saved: p1_llama_3.1_8b_instant.txt
✅ Saved: p1_llama_3.3_70b_versatile.txt
✅ Saved: p2_llama_3.1_8b_instant.txt
✅ Saved: p2_llama_3.3_70b_versatile.txt
✅ Saved: p3_llama_3.1_8b_instant.txt
✅ Saved: p3_llama_3.3_70b_versatile.txt
✅ Saved: p4_llama_3.1_8b_instant.txt
✅ Saved: p4_llama_3.3_70b_versatile.txt
✅ Saved: p5_llama_3.1_8b_instant.txt
✅ Saved: p5_llama_3.3_70b_versatile.txt

💾 All full responses saved to: C:\Users\Farhan\Desktop\Experiment_Thesis\data\pilot\micro_test\full_responses


In [36]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("C:/Users/Farhan/Desktop/Experiment_Thesis/data/pilot/micro_test")

# Load the scored results
score_df = pd.read_parquet(DATA_DIR / "micro_scores.parquet")

# Select only the columns needed for analysis
metrics_df = score_df[[
    "prompt_id", "prompt_type", "model", "n_valid", 
    "mean_pairwise_sim", "ds_raw", "p_factual_inverted"
]].copy()

# Add a human-readable expectation column
def expected_outcome(row):
    if row["prompt_type"] in ["factual_simple", "factual_medical"]:
        return "HIGH P(factual) expected (correct answers)"
    elif row["prompt_type"] in ["adversarial_trick", "adversarial_misconception"]:
        return "LOW P(factual) expected IF model repeats myth; HIGH if it corrects"
    elif row["prompt_type"] == "ambiguous_history":
        return "MODERATE-HIGH P(factual) expected (legitimate variation OK)"
    else:
        return "Unknown"

metrics_df["expected"] = metrics_df.apply(expected_outcome, axis=1)

# Export to CSV (easy to copy-paste into Excel/Sheets)
csv_path = DATA_DIR / "metrics_summary.csv"
metrics_df.to_csv(csv_path, index=False)

print("📊 METRICS SUMMARY (copy-paste friendly):")
print(metrics_df.to_string())
print(f"\n💾 Saved to: {csv_path}")

📊 METRICS SUMMARY (copy-paste friendly):
  prompt_id                prompt_type                    model  n_valid  mean_pairwise_sim    ds_raw  p_factual_inverted                                                            expected
0        p1             factual_simple     llama-3.1-8b-instant       10           0.959830  1.000000            0.000000                          HIGH P(factual) expected (correct answers)
1        p1             factual_simple  llama-3.3-70b-versatile       10           0.996073  1.000000            0.000000                          HIGH P(factual) expected (correct answers)
2        p2            factual_medical     llama-3.1-8b-instant       10           0.932460  0.955556            0.044444                          HIGH P(factual) expected (correct answers)
3        p2            factual_medical  llama-3.3-70b-versatile       10           0.952258  1.000000            0.000000                          HIGH P(factual) expected (correct answers)
4        